[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AltamarMx/anomalias-2026-2/blob/main/notebooks/017_anomalias_teoria.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)
from IPython.display import display, Markdown

# Semana 9A — Detección de Anomalías: Métodos Estadísticos (Teoría)

**Objetivo:** Entender qué es un outlier en contexto meteorológico,
implementar detectores basados en z-score, MAD e IQR, y comprender
sus limitaciones con datos estacionales.

**Temas:**

1. ¿Qué es un outlier?
2. Z-score y sus limitaciones
3. Métodos robustos: MAD e IQR
4. Z-score estacional

---
## 1. ¿Qué es un outlier?

Un outlier es una observación que se desvía significativamente de
lo esperado.  Pero **"esperado" depende del contexto**.

En datos meteorológicos, tres categorías:

| Tipo | Ejemplo | Acción |
|:---|:---|:---|
| **(a) Error de sensor** | Spike puntual, "stuck sensor" | Eliminar e imputar |
| **(b) Físicamente imposible** | `rh < 0`, `p_atm = 443 hPa` | Eliminar (reglas deterministas) |
| **(c) Evento extremo real** | Tormenta, ola de calor | **Conservar** — es dato válido |

El desafío: distinguir (a) y (b) de (c).  No hay algoritmo perfecto —
siempre se requiere **juicio del dominio**.

---
## 2. Z-score

$$z_i = \frac{x_i - \mu}{\sigma}$$

Marcar como outlier si $|z_i| > k$ (típicamente $k = 3$).

### 2.1 Caso simple: datos normales con outliers inyectados

In [ ]:
_n = 500
_datos = rng.normal(0, 1, _n)
# Inyectar 5 outliers
_idx_out = rng.choice(_n, 5, replace=False)
_datos[_idx_out] = rng.choice([-5, 5], 5) + rng.normal(0, 0.3, 5)

_mu = _datos.mean()
_sigma = _datos.std()
_z = (_datos - _mu) / _sigma
_es_outlier = np.abs(_z) > 3

fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(12, 4))

# Serie
ax1a.scatter(range(_n), _datos, c="steelblue", s=8, alpha=0.5)
ax1a.scatter(
    np.where(_es_outlier)[0], _datos[_es_outlier],
    c="crimson", s=40, zorder=5, label=f"Outliers ({_es_outlier.sum()})",
)
ax1a.axhline(_mu + 3 * _sigma, color="crimson", ls="--", lw=1, alpha=0.6)
ax1a.axhline(_mu - 3 * _sigma, color="crimson", ls="--", lw=1, alpha=0.6)
ax1a.set_xlabel("Índice")
ax1a.set_ylabel("Valor")
ax1a.set_title("Datos normales + 5 outliers inyectados")
ax1a.legend()

# Histograma
ax1b.hist(_datos, bins=40, density=True, color="steelblue",
          alpha=0.7, edgecolor="white")
ax1b.axvline(_mu + 3 * _sigma, color="crimson", ls="--", lw=1.5)
ax1b.axvline(_mu - 3 * _sigma, color="crimson", ls="--", lw=1.5)
ax1b.set_xlabel("Valor")
ax1b.set_ylabel("Densidad")
ax1b.set_title("Distribución con umbrales ±3σ")

plt.tight_layout()
plt.show()

> Con datos normales, el z-score funciona bien: los 5 outliers
> inyectados caen fuera de ±3σ.

### 2.2 El problema: datos multimodales / estacionales

In [ ]:
# Simular mezcla de dos estaciones: invierno N(15,3) y verano N(30,3)
_invierno = rng.normal(15, 3, 300)
_verano = rng.normal(30, 3, 300)
_mezcla = np.concatenate([_invierno, _verano])
rng.shuffle(_mezcla)

# Inyectar 3 outliers reales
_outlier_idx = [10, 250, 500]
_outlier_vals = [2, 45, 22.5]  # fuera de ambas estaciones
for i, v in zip(_outlier_idx, _outlier_vals):
    _mezcla[i] = v

_mu = _mezcla.mean()
_sigma = _mezcla.std()
_z = (_mezcla - _mu) / _sigma
_es_outlier = np.abs(_z) > 3

fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(12, 4))

ax2a.scatter(range(len(_mezcla)), _mezcla, c="steelblue", s=8, alpha=0.5)
ax2a.scatter(
    np.where(_es_outlier)[0], _mezcla[_es_outlier],
    c="crimson", s=40, zorder=5,
    label=f"z-score detectó {_es_outlier.sum()}",
)
ax2a.axhline(_mu + 3 * _sigma, color="crimson", ls="--", lw=1, alpha=0.6)
ax2a.axhline(_mu - 3 * _sigma, color="crimson", ls="--", lw=1, alpha=0.6)
ax2a.set_title("Mezcla invierno/verano + 3 outliers reales")
ax2a.set_xlabel("Índice")
ax2a.set_ylabel("Temperatura (°C)")
ax2a.legend(fontsize=9)

ax2b.hist(_mezcla, bins=50, density=True, color="steelblue",
          alpha=0.7, edgecolor="white")
ax2b.axvline(_mu + 3 * _sigma, color="crimson", ls="--", lw=1.5)
ax2b.axvline(_mu - 3 * _sigma, color="crimson", ls="--", lw=1.5)
ax2b.set_xlabel("Temperatura (°C)")
ax2b.set_ylabel("Densidad")
ax2b.set_title("Distribución bimodal — z-score global")

plt.tight_layout()
plt.show()

> **Problema:**
> - La media global está ~22.5 °C (entre las dos estaciones).
> - σ es grande (~8°C) porque incluye la separación entre modos.
> - El umbral ±3σ queda en ~−1°C y ~46°C — **demasiado amplio**.
> - Valores normales de verano (35°C) no se detectan como outliers,
>   pero tampoco los outliers reales moderados (2°C, 45°C).
>
> **Conclusión:** El z-score global **falla** con datos estacionales
> o multimodales.

---
## 3. Métodos robustos: MAD e IQR

El z-score usa $\mu$ y $\sigma$, que son **sensibles a outliers extremos**.
Un solo valor de 1000 puede inflar $\sigma$ tanto que ningún otro punto
parece anómalo.

### MAD (Median Absolute Deviation)

$$\text{MAD} = \text{mediana}\left(|x_i - \widetilde{x}|\right)$$

Umbral: outlier si $\frac{|x_i - \widetilde{x}|}{1.4826 \cdot \text{MAD}} > k$

(El factor 1.4826 hace que MAD sea equivalente a σ bajo normalidad.)

### IQR (Interquartile Range)

$$\text{IQR} = Q_3 - Q_1$$

Outlier si $x < Q_1 - 1.5 \cdot \text{IQR}$ o $x > Q_3 + 1.5 \cdot \text{IQR}$

(Son los bigotes del boxplot.)

In [ ]:
# Datos normales + un outlier extremo
_datos = rng.normal(25, 3, 200)
_datos = np.append(_datos, 200)  # outlier extremo

_mu = _datos.mean()
_sigma = _datos.std()
_mediana = np.median(_datos)
_mad = np.median(np.abs(_datos - _mediana))
_mad_scaled = 1.4826 * _mad
_q1, _q3 = np.percentile(_datos, [25, 75])
_iqr = _q3 - _q1

# Umbrales
_z_lo, _z_hi = _mu - 3 * _sigma, _mu + 3 * _sigma
_mad_lo = _mediana - 3 * _mad_scaled
_mad_hi = _mediana + 3 * _mad_scaled
_iqr_lo = _q1 - 1.5 * _iqr
_iqr_hi = _q3 + 1.5 * _iqr

# Detecciones
_out_z = np.abs((_datos - _mu) / _sigma) > 3
_out_mad = np.abs(_datos - _mediana) / _mad_scaled > 3
_out_iqr = (_datos < _iqr_lo) | (_datos > _iqr_hi)

fig3, ax3 = plt.subplots(figsize=(12, 5))

ax3.scatter(range(len(_datos)), _datos, c="steelblue", s=15, alpha=0.5)

_methods = [
    ("Z-score (±3σ)", _z_lo, _z_hi, "crimson", _out_z.sum()),
    ("MAD (±3·MAD)", _mad_lo, _mad_hi, "darkorange", _out_mad.sum()),
    ("IQR (Tukey)", _iqr_lo, _iqr_hi, "seagreen", _out_iqr.sum()),
]

for nombre, lo, hi, color, n_det in _methods:
    ax3.axhline(hi, color=color, ls="--", lw=1.5, alpha=0.7,
                 label=f"{nombre}: {n_det} detectados")
    ax3.axhline(lo, color=color, ls="--", lw=1.5, alpha=0.7)

ax3.set_xlabel("Índice")
ax3.set_ylabel("Valor")
ax3.set_title(
    f"Efecto de un outlier extremo (x=200) en los umbrales  "
    f"(σ={_sigma:.1f},  MAD·1.48={_mad_scaled:.1f})"
)
ax3.legend(loc="upper left")
ax3.set_ylim(-5, 210)
plt.tight_layout()
plt.show()

> **Observa:**
> - El outlier extremo (x=200) **infla σ a ~15**, haciendo que el umbral
>   z-score suba a ~70 → no detecta nada más.
> - **MAD** usa la mediana, que es inmune al outlier → mantiene un umbral
>   sensato (~34) y detecta el punto extremo.
> - **IQR** también es robusto: los cuartiles no se mueven con un solo
>   punto extremo.
>
> **Lección:** Para datos con outliers severos (sensores meteorológicos
> descalibrados), usar MAD o IQR en lugar de z-score.

---
## 4. Z-score estacional

La solución al problema de la sección 2: en vez de calcular z sobre
**toda** la serie, agrupar por **(hora del día, mes)** y calcular z
dentro de cada grupo.

Así, "35°C a las 3PM en julio" se compara con otros valores de las
3PM en julio — no con "5°C a las 4AM en enero".

In [ ]:
# Señal sintética con estacionalidad
_n = 730
_t = np.arange(_n)
_estacionalidad = 10 * np.sin(2 * np.pi * _t / 365)
_ruido = rng.normal(0, 1.5, _n)
_y = 22 + _estacionalidad + _ruido

# Inyectar 5 outliers
_outlier_idx = np.array([50, 180, 300, 450, 600])
_y[_outlier_idx] += rng.choice([-12, 12], len(_outlier_idx))

# Z-score global
_z_global = (_y - _y.mean()) / _y.std()
_out_global = np.abs(_z_global) > 3

# Z-score estacional (agrupar por "posición en el ciclo")
_periodo = 365
_grupo = _t % _periodo
_z_estac = np.zeros(_n)
for g in np.unique(_grupo):
    _mask = _grupo == g
    _vals = _y[_mask]
    _z_estac[_mask] = (_vals - _vals.mean()) / _vals.std()
_out_estac = np.abs(_z_estac) > 3

fig4, (ax4a, ax4b) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Global
ax4a.plot(_t, _y, "steelblue", lw=0.6)
ax4a.scatter(
    _t[_out_global], _y[_out_global],
    c="crimson", s=40, zorder=5,
    label=f"Z-score global: {_out_global.sum()} detecciones",
)
ax4a.scatter(
    _outlier_idx, _y[_outlier_idx],
    facecolors="none", edgecolors="black", s=80, lw=2, zorder=6,
    label="Outliers reales (inyectados)",
)
ax4a.set_ylabel("Valor")
ax4a.set_title("Z-score global → falsos positivos estacionales")
ax4a.legend(fontsize=9)

# Estacional
ax4b.plot(_t, _y, "steelblue", lw=0.6)
ax4b.scatter(
    _t[_out_estac], _y[_out_estac],
    c="darkorange", s=40, zorder=5,
    label=f"Z-score estacional: {_out_estac.sum()} detecciones",
)
ax4b.scatter(
    _outlier_idx, _y[_outlier_idx],
    facecolors="none", edgecolors="black", s=80, lw=2, zorder=6,
    label="Outliers reales",
)
ax4b.set_xlabel("Día")
ax4b.set_ylabel("Valor")
ax4b.set_title("Z-score estacional → detección limpia")
ax4b.legend(fontsize=9)

plt.tight_layout()
plt.show()

> **Resultado:**
> - **Z-score global** marca muchos valores normales de los extremos
>   estacionales como outliers (falsos positivos) y puede perder
>   outliers moderados.
> - **Z-score estacional** compara cada valor con su contexto temporal
>   → detecta los 5 outliers reales sin falsos positivos.
>
> En el laboratorio aplicaremos esto agrupando por (hora, mes) sobre
> el dataset ClimaLab.

---
## Resumen

| Método | Fórmula | Fortaleza | Debilidad |
|:---|:---|:---|:---|
| **Z-score** | $(x - \mu) / \sigma$ | Simple, intuitivo | Sensible a outliers extremos y estacionalidad |
| **MAD** | $\|x - \widetilde{x}\| / (1.48 \cdot \text{MAD})$ | Robusto a outliers extremos | No maneja estacionalidad |
| **IQR** | Tukey (1.5·IQR) | Robusto, equivale al boxplot | No maneja estacionalidad |
| **Z-score estacional** | z por grupo (hora, mes) | Maneja estacionalidad | Requiere suficientes datos por grupo |

**Próxima sesión (9B):** Aplicar todos estos métodos al dataset
ClimaLab y comparar sus resultados sistemáticamente.